# ScaDS.AI Coding

This notebook rates posts stored in `data.csv` using the ScaDS.AI API and a system prompt stored in `systemprompt.md`.

Official examples on how to access ScaDS.AI LLM Service via the API are found here: 
https://gitlab.hrz.tu-chemnitz.de/scads-ai-llm/scads-ai-llm-api-examples

In [ ]:
import pandas as pd
import os
import json
import time
import json_repair  # tolerant JSON parser: fixes unescaped quotes, trailing commas, etc.
from openai import OpenAI

my_api_key = open("apikey.txt", "r").readline().strip()

# Load Data

All posts to be rated live in a single `data.csv` (must contain at least `id` and `text` columns).

In [2]:
d = pd.read_csv("data.csv")
d = d.drop_duplicates(subset="id")
d.head()

,id,user_name,text
0,1,MdB_Beispiel1,"Die Menschen in unserem Land haben es satt, vo..."
1,2,MdB_Beispiel2,Heute im Ausschuss über die Reform des Vergabe...
2,3,MdB_Beispiel3,Die sogenannten Experten in ihren Elfenbeintür...
3,4,MdB_Beispiel4,"Freue mich, morgen beim Bürgerdialog in meinem..."
4,5,MdB_Beispiel5,Die Lügenpresse und die korrupten Eliten wolle...


# ScaDS.AI API

In [3]:
# read prompt file
with open("systemprompt.md", "r", encoding="utf-8") as f:
    system_prompt = f.read()

# initiate model
client = OpenAI(base_url="https://llm.scads.ai/v1", api_key=my_api_key)
model_name = "Qwen/Qwen3-Coder-30B-A3B-Instruct"

## Process Single Post
Let's test the API access with a single post.

In [ ]:
response = client.chat.completions.create(
    # The developed prompt is included as a system prompt, thereby
    # ovevrwriting the LLMs inherent instructions to be a nice
    # chatbot-assistant and focusing it only on the coding task
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": d["text"][0]}
    ],
    model=model_name
)

answer = response.choices[0].message.content
print(answer)

# json_repair tolerates the malformed JSON the model sometimes returns
# (e.g. unescaped double quotes inside string values).
answer_json = answer[answer.find("{"):answer.rfind("}") + 1]
data = json_repair.loads(answer_json)
data

## Process Multiple Posts

In [ ]:
# Output fields expected from the model's JSON response/Your Rating Scale.
# These are initialized for every row so the resulting DataFrame always
# has a consistent set of columns, even when parsing fails.
output_fields = [
    "holistic_redescription",
    "actor_analysis",
    "people_explanation",
    "people_score",
    "elite_explanation",
    "elite_score",
    "antagonism_explanation",
    "antagonism_score",
]

d_results = []

for row in d.itertuples():
    print(f"Processing {row.Index + 1}/{len(d)} | ID: {row.id}...")
    
    # Wait for two seconds. 
    # This prevents us from DDOSing the TUD Server 
    # (we want to be kind to avoid getting rate limited or blocked)
    time.sleep(2)

    # Initialize the result with id, text and all output fields set to None.
    row_result = {"id": row.id, "text": row.text, "error": None}
    row_result.update({field: None for field in output_fields})

    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": row.text}
        ],
        model=model_name
    )

    answer = response.choices[0].message.content

    # Parse the json output by searching for brackets. json_repair fixes the
    # malformed JSON the model sometimes returns (e.g. unescaped quotes).
    answer_json_str = answer[answer.find("{"):answer.rfind("}") + 1]
    answer_data = json_repair.loads(answer_json_str)

    if isinstance(answer_data, dict) and answer_data:
        # Only keep known output fields to avoid unexpected columns.
        for field in output_fields:
            row_result[field] = answer_data.get(field)
    else:
        print(f"Error parsing JSON for ID {row.id}")
        row_result["error"] = "JSON Parse Error"

    d_results.append(row_result)

d_scored = pd.DataFrame(d_results, columns=["id", "text"] + output_fields + ["error"])

print(d_scored.head())

# Save Results

Write the scored posts to a timestamped CSV so previous runs are not overwritten.

In [ ]:
timestamp = time.strftime("%Y%m%d_%H%M%S")
output_path = f"data_scored_{timestamp}.csv"

d_scored.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved {len(d_scored)} rows to {output_path}")